# ST446 Project: Feature Engineering

This notebook prepares the cleaned flight delay dataset for modelling. It encodes categorical 
features, assembles the feature vector, and writes a modelling-ready parquet file containing 
the feature vector alongside all three target columns (`delay_class`, `is_late`, `delay_severity`).

**Features included:**
- `carrier_id` — low cardinality (18 values); encoded with StringIndexer + OneHotEncoder
- `dest` — low cardinality (3 values: ORD, JFK, ATL); encoded with StringIndexer + OneHotEncoder
- `origin` — high cardinality (~241 unique airports); encoded with FeatureHasher to avoid
  sparse 241-dimension OHE vectors. Hash size set to 64 — sufficient to minimise collisions
  at this cardinality while keeping the feature vector compact.
- Numeric (kept as-is): `dep_hour`, `day_of_week`, `month`, `distance`

**Excluded from features:**
- `distance_group` — r=0.92 with `distance`; dropped to avoid multicollinearity, particularly
  for Logistic Regression which assumes no perfect multicollinearity among predictors
- `arr_delay` — target is derived from this; including it would be data leakage
- `year` — excluded to avoid learning COVID-era anomalies that won't generalise
- Delay cause columns (`delay_carrier`, `delay_weather`, etc.) — only observable post-departure; data leakage
- `delay_none` — post-departure indicator, same leakage concern

**Targets preserved (not in feature vector):**
- `delay_class` — original 4-class target (0=No, 1=Minor, 2=Major, 3=Severe)
- `is_late` — Layer 1 binary target (0=On-time ≤15min, 1=Late >15min)
- `delay_severity` — Layer 2 binary target among late flights (0=Minor 15–45min, 1=Major >45min)

## 1. Setup

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, FeatureHasher
)
import os

spark = SparkSession.builder.appName("ST446-Features").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/28 11:24:22 WARN Utils: Your hostname, MacBook-Pro-3.local, resolves to a loopback address: 127.0.0.1; using 10.254.86.222 instead (on interface en0)
26/04/28 11:24:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/28 11:24:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1


## 2. Load Data

In [2]:
df = spark.read.parquet("data/df_final_clean.parquet")
print(f"Rows: {df.count():,} | Columns: {len(df.columns)}")
df.show(3, truncate=False)

Rows: 3,480,548 | Columns: 17
+----------+------+----+--------+----+-----------+-----+--------+--------------+---------+-------------+-------------+---------+--------------+-------------------+----------+-----------+
|carrier_id|origin|dest|dep_hour|year|day_of_week|month|distance|distance_group|arr_delay|delay_carrier|delay_weather|delay_nas|delay_security|delay_late_aircraft|delay_none|delay_class|
+----------+------+----+--------+----+-----------+-----+--------+--------------+---------+-------------+-------------+---------+--------------+-------------------+----------+-----------+
|19393     |AUS   |ATL |6       |2022|7          |1    |813.0   |1             |-15.0    |0            |0            |0        |0             |0                  |1         |0          |
|19393     |AUS   |ATL |18      |2022|7          |1    |813.0   |1             |8.0      |0            |0            |0        |0             |0                  |0         |1          |
|19393     |BNA   |ATL |22      |20

## 3. Derive Target Columns

We derive two binary target columns alongside the existing `delay_class`:
- `is_late`: Layer 1 target — on-time (≤15 min) vs late (>15 min)
- `delay_severity`: Layer 2 target — minor delay (15–45 min) vs major delay (>45 min)

Note: `delay_severity` is only meaningful for late flights (where `is_late == 1`). 
For on-time flights it defaults to 0 and is masked during Layer 2 training.

In [3]:
df = df.withColumn("is_late", (col("arr_delay") > 15).cast("int"))
df = df.withColumn("delay_severity",
    when(col("arr_delay") <= 45, 0).otherwise(1))

# Verify
print("Layer 1 distribution (is_late):")
df.groupBy("is_late").count().orderBy("is_late").show()

print("Layer 2 distribution (delay_severity, late flights only):")
df.filter(col("is_late") == 1).groupBy("delay_severity").count().orderBy("delay_severity").show()

Layer 1 distribution (is_late):


+-------+-------+
|is_late|  count|
+-------+-------+
|      0|2933037|
|      1| 547511|
+-------+-------+

Layer 2 distribution (delay_severity, late flights only):
+--------------+------+
|delay_severity| count|
+--------------+------+
|             0|281459|
|             1|266052|
+--------------+------+



## 4. Drop Excluded Columns

Remove columns that must not appear in the feature vector. Target columns are retained.

In [4]:
cols_to_drop = [
    "arr_delay",           # target is derived from this
    "distance_group",      # r=0.92 with distance; dropped to avoid multicollinearity
    "delay_carrier",       # post-departure leakage
    "delay_weather",       # post-departure leakage
    "delay_nas",           # post-departure leakage
    "delay_security",      # post-departure leakage
    "delay_late_aircraft", # post-departure leakage
    "delay_none",          # post-departure leakage
]

df = df.drop(*cols_to_drop)
print("Remaining columns:", df.columns)

Remaining columns: ['carrier_id', 'origin', 'dest', 'dep_hour', 'year', 'day_of_week', 'month', 'distance', 'delay_class', 'is_late', 'delay_severity']


## 5. Encode Categorical Features

Encoding strategy differs by cardinality:
- `carrier_id` and `dest` are low cardinality (18 and 3 values respectively) — StringIndexer +
  OneHotEncoder is appropriate and produces interpretable, lossless encodings.
- `origin` has ~241 unique values — OneHotEncoder would produce a 240-dimension sparse vector,
  significantly increasing dimensionality and training time. FeatureHasher maps origin codes
  to a fixed 64-dimension dense vector via hashing, trading a small risk of collision for a
  substantial reduction in feature space. At 241 categories, a hash size of 64 gives acceptable
  collision probability while reducing origin's contribution to the vector from 240 to 64 dimensions.

In [5]:
# OHE for low-cardinality categoricals: carrier_id and dest
low_card_cols = ["carrier_id", "dest"]

indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_idx",
        handleInvalid="keep"
    )
    for c in low_card_cols
]

encoders = [
    OneHotEncoder(
        inputCol=f"{c}_idx",
        outputCol=f"{c}_ohe"
    )
    for c in low_card_cols
]

# FeatureHasher for high-cardinality origin (~241 unique airports)
# numFeatures=64 keeps the vector compact while minimising hash collisions
origin_hasher = FeatureHasher(
    inputCols=["origin"],
    outputCol="origin_hashed",
    numFeatures=64
)

print("Encoding: OHE for", low_card_cols)
print("Encoding: FeatureHasher (64 buckets) for origin")

Encoding: OHE for ['carrier_id', 'dest']
Encoding: FeatureHasher (64 buckets) for origin


## 6. Assemble Feature Vector

VectorAssembler combines all feature columns into a single `features` vector column, 
which is the required input format for Spark MLlib models.

In [6]:
numeric_cols = ["dep_hour", "day_of_week", "month", "distance"]
ohe_cols = [f"{c}_ohe" for c in low_card_cols]

assembler = VectorAssembler(
    inputCols=numeric_cols + ohe_cols + ["origin_hashed"],
    outputCol="features",
    handleInvalid="keep"
)

print("Feature vector will include:")
print("  Numeric:", numeric_cols)
print("  OHE:", ohe_cols)
print("  Hashed: origin_hashed (64 dims)")

Feature vector will include:
  Numeric: ['dep_hour', 'day_of_week', 'month', 'distance']
  OHE: ['carrier_id_ohe', 'dest_ohe']
  Hashed: origin_hashed (64 dims)


## 7. Fit and Apply Pipeline

In [7]:
pipeline = Pipeline(stages=indexers + encoders + [origin_hasher, assembler])

print("Fitting pipeline...")
pipeline_model = pipeline.fit(df)

print("Transforming data...")
df_features = pipeline_model.transform(df)

# Keep only the columns needed for modelling
final_cols = ["features", "year", "delay_class", "is_late", "delay_severity"]
df_final = df_features.select(final_cols)

print("\nFinal schema:")
df_final.printSchema()
df_final.show(3, truncate=True)

Fitting pipeline...


Transforming data...

Final schema:
root
 |-- features: vector (nullable = true)
 |-- year: integer (nullable = true)
 |-- delay_class: integer (nullable = true)
 |-- is_late: integer (nullable = true)
 |-- delay_severity: integer (nullable = false)

+--------------------+----+-----------+-------+--------------+
|            features|year|delay_class|is_late|delay_severity|
+--------------------+----+-----------+-------+--------------+
|(89,[0,1,2,3,11,2...|2022|          0|      0|             0|
|(89,[0,1,2,3,11,2...|2022|          1|      0|             0|
|(89,[0,1,2,3,11,2...|2022|          1|      1|             1|
+--------------------+----+-----------+-------+--------------+
only showing top 3 rows


## 8. Verify Feature Vector

In [8]:
from pyspark.ml.functions import vector_to_array

# Check feature vector size
sample = df_final.select(vector_to_array("features").alias("f")).first()
print(f"Feature vector length: {len(sample['f'])}")

# Confirm no nulls in features or targets
from pyspark.sql.functions import sum as spark_sum
null_check = df_final.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in ["delay_class", "is_late", "delay_severity"]
])
print("\nNull counts in target columns:")
null_check.show()

# Confirm target distributions are preserved
print("delay_class distribution:")
df_final.groupBy("delay_class").count().orderBy("delay_class").show()

print("is_late distribution:")
df_final.groupBy("is_late").count().orderBy("is_late").show()

print("delay_severity distribution (all rows):")
df_final.groupBy("delay_severity").count().orderBy("delay_severity").show()

Feature vector length: 89

Null counts in target columns:
+-----------+-------+--------------+
|delay_class|is_late|delay_severity|
+-----------+-------+--------------+
|          0|      0|             0|
+-----------+-------+--------------+

delay_class distribution:
+-----------+-------+
|delay_class|  count|
+-----------+-------+
|          0|2444610|
|          1| 829685|
|          2| 114931|
|          3|  91322|
+-----------+-------+

is_late distribution:
+-------+-------+
|is_late|  count|
+-------+-------+
|      0|2933037|
|      1| 547511|
+-------+-------+

delay_severity distribution (all rows):
+--------------+-------+
|delay_severity|  count|
+--------------+-------+
|             0|3214496|
|             1| 266052|
+--------------+-------+



## 9. Save Modelling-Ready Dataset

In [9]:
output_path = "data/df_features.parquet"

df_final.write.mode("overwrite").parquet(output_path)
print(f"Saved to {output_path}")

# Quick reload check
df_check = spark.read.parquet(output_path)
print(f"Reload check — rows: {df_check.count():,} | columns: {df_check.columns}")

Saved to data/df_features.parquet
Reload check — rows: 3,480,548 | columns: ['features', 'year', 'delay_class', 'is_late', 'delay_severity']


In [10]:
# THIS STAYS AT THE END
spark.stop()